[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/27_vit_patch_solution.ipynb)

# ✅ Solution: vit_patch

Implement the **patch embedding** layer from Vision Transformer (ViT).

### Signature
```python
class PatchEmbedding(nnx.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, C, H, W)
        # Returns: (B, num_patches, embed_dim)
```

### Algorithm
1. Reshape image into non-overlapping patches: `(B, C, H, W)` → `(B, N, C*P*P)`
2. Project each patch: `nn.Linear(C*P*P, embed_dim)`
3. `num_patches = (img_size // patch_size) ** 2`


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import math


In [ ]:
# ✅ SOLUTION

import jax.numpy as jnp
from flax import nnx
class PatchEmbedding(nnx.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim, *, rngs):
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nnx.Linear(in_channels * patch_size * patch_size, embed_dim, rngs=rngs)
    def __call__(self, x_BCHW):
        B, C, H, W = x_BCHW.shape
        P = self.patch_size
        patches_BPD = (
            x_BCHW.reshape(B, C, H // P, P, W // P, P)
            .transpose(0, 2, 4, 1, 3, 5)
            .reshape(B, -1, C * P * P)
        )
        return self.proj(patches_BPD)


In [ ]:
# Verify
print(PatchEmbedding)


In [ ]:
from jax_judge import check
check("vit_patch")
